# Tutorial 1: Environments and Neural Networks

**zrth** extracts symbolic reactive modules from Python classes. You write a standard [Gymnasium](https://gymnasium.farama.org/) environment or a [PyTorch](https://pytorch.org/) neural network, then wrap it with `Env()` or `NN()` — zrth analyzes it and produces a formal model with typed wires and terms that can be used for verification or training.

This tutorial walks through:
1. Defining an environment (plain `gym.Env`) and wrapping it with `Env()`
2. Defining a neural network (plain `nn.Module`) and wrapping it with `NN()`
3. Inspecting the extracted modules

## Defining an environment

Write a standard [Gymnasium](https://gymnasium.farama.org/) environment by subclassing `gym.Env`:

- **`__init__`**: set `action_space`, `observation_space`, any parameters (e.g. `self.y0`), and state variables (e.g. `self.x`)
- **`reset`**: initialize all state variables, return `(observation, info)`
- **`step`**: update state given an action, return `(observation, reward, terminated, truncated, info)`

**Important:** state variables must also be assigned in `__init__` so the analyzer can infer their types. For example, even though `reset` sets `self.x = 0.0`, you should also write `self.x = 0.0` in `__init__`.

Below is a counter system with three variables $x$, $y$, $z$. The variable $x$ increments while $x < y \lor x < z$, and resets to $0$ otherwise. We want to show that $x = y \lor x = z$ must occur infinitely often.

In [7]:
import torch
import torch.nn as nn
import gymnasium as gym
from gymnasium import spaces
from zrth import Env, NN


class CounterEnv(gym.Env):
    """Counter environment: x increments while x < y or x < z, resets otherwise."""

    def __init__(self, y0=5.0, z0=3.0):
        super().__init__()

        self.action_space = spaces.Discrete(1)
        self.observation_space = spaces.Box(low=-1e6, high=1e6, shape=(1,))

        self.y0 = y0
        self.z0 = z0

        # Also set state variables here so the analyzer can infer their dtype
        self.x = 0.0
        self.y = y0
        self.z = z0

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.x = 0.0
        self.y = self.y0
        self.z = self.z0

        observation = self.x
        return observation, {}

    def step(self, action):
        if self.x < self.y or self.x < self.z:
            self.x = self.x + 1.0
        else:
            self.x = 0.0

        # y and z remain constant
        self.y = self.y
        self.z = self.z

        at_target = self.x == self.y or self.x == self.z
        reward = 1.0 if at_target else 0.0
        terminated = at_target
        truncated = False
        observation = self.x
        return observation, reward, terminated, truncated, {}

## Wrapping and inspection

When you call `Env(counter_instance)`, zrth analyzes the class's `__init__`, `reset`, and `step` methods. It:
1. Infers action/observation types from the gymnasium spaces
2. Classifies `self.*` attributes as **private** (read+write state) or **parameters** (read-only constants)
3. Creates typed wire pairs for each variable and signal
4. Converts `reset` into **init** terms and `step` into **update** terms

Print the extracted module to see the result.

In [8]:
counter = CounterEnv(y0=5.0, z0=3.0)
wrapped_counter = Env(counter)
print(wrapped_counter)

module
  external
    w56, w57 : Float(1)
  interface
    w58, w59 : Float(1)
    w60, w61 : Float(1)
    w62, w63 : Bool(1)
    w64, w65 : Bool(1)
  private
    w48, w49 : Float(1)
    w50, w51 : Float(1)
    w52, w53 : Float(1)
  atom controls w49, w51, w53, w59, w61, w63, w65 reads w48, w50, w52
  init
    Tensor([5]) w54; 
    Tensor([3]) w55; 
    Tensor([0]) w66; 
    Id w53; w66
    Id w49; w54
    Id w51; w55
    Id w59; w66
    Tensor([0]) w61; 
    Const: false w63; 
    Const: false w65; 
  update
    Tensor([5]) w54; 
    Tensor([3]) w55; 
    Lt w67; w52, w48
    Lt w68; w52, w50
    Tensor([0]) w69; 
    Tensor([1]) w70; 
    Ite w71; w67, w70, w68
    Tensor([1]) w72; 
    Add w73; w52, w72
    Tensor([0]) w74; 
    Ite w75; w71, w73, w74
    Id w53; w75
    Id w49; w48
    Id w51; w50
    Eq w76; w75, w48
    Eq w77; w75, w50
    Tensor([0]) w78; 
    Tensor([1]) w79; 
    Ite w80; w76, w79, w77
    Tensor([1]) w81; 
    Tensor([0]) w82; 
    Ite w83; w80, w81, w82
    

## Reading the output

The printed module shows:

- **external**: input wires — here, the action (unused by this environment, but still present)
- **interface**: output wires — observation, reward, terminated, truncated
- **private**: internal state — wire pairs `wN, wM` for x, y, z (each pair is `latched, next`)
- **init**: terms from `reset` — how each wire is initialized
- **update**: terms from `step` — how each wire is updated, including `Ite` (if-then-else), `Lt`, `Eq`, `Add`

## Defining a neural network

Write a standard `nn.Module`, then wrap it with `NN(instance)` to extract a symbolic module:

- **`__init__`**: define `nn.Linear` layers
- **`forward`**: standard PyTorch forward pass

zrth extracts the architecture as a **combinatorial** (stateless) module. The resulting module has:
- **extl** (external input) wire pair — sized from the first layer's `in_features`
- **intf** (interface output) wire pair — sized from the last layer's `out_features`

The ranking function below maps $(x, y, z) \in \mathbb{R}^3$ to a scalar. It is used in formal verification to prove that the liveness property $x = y \lor x = z$ holds infinitely often — not a controller, so we don't compose it with the counter.

In [9]:
class RankingNN(nn.Module):
    """Ranking function: R(x, y, z) -> scalar.

    Architecture: [3] -> 2 -> [1]
    """

    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(3, 2)
        self.fc2 = nn.Linear(2, 1)

    def forward(self, state):
        x = torch.relu(self.fc1(state))
        return self.fc2(x)

Wrap and inspect. Note that the NN module has no `private` wires (it's stateless) and the `init`/`update` blocks are identical (combinatorial — same computation every step).

In [10]:
ranking = RankingNN()
wrapped_ranking = NN(ranking)
print(wrapped_ranking)

module
  external
    w85, w86 : Float(3)
  interface
    w87, w88 : Float(1)
  atom controls w88 awaits w86
  init
    Tensor([-0.36338144540786743 -0.24140074849128723 -0.2480044811964035 0.32741886377334595 0.21225106716156006 ...]) w89; 
    Tensor([-0.013416432775557041 -0.18645182251930237]) w90; 
    Linear w91; w86, w89, w90
    ReLU w92; w91
    Tensor([-0.6736899614334106 0.12457335740327835]) w93; 
    Tensor([-0.5915284156799316]) w94; 
    Linear w95; w92, w93, w94
    Id w88; w95
  update
    Tensor([-0.36338144540786743 -0.24140074849128723 -0.2480044811964035 0.32741886377334595 0.21225106716156006 ...]) w89; 
    Tensor([-0.013416432775557041 -0.18645182251930237]) w90; 
    Linear w91; w86, w89, w90
    ReLU w92; w91
    Tensor([-0.6736899614334106 0.12457335740327835]) w93; 
    Tensor([-0.5915284156799316]) w94; 
    Linear w95; w92, w93, w94
    Id w88; w95



## Training the ranking function

A **ranking function** $R: \mathcal{S} \to \mathbb{R}$ can be used to prove that a liveness property holds infinitely often. The key conditions are:

1. $R(s) \geq 0$ for all states $s$
2. $R(s) = 0$ when the liveness property holds (here: $x = y \lor x = z$)
3. $R(s') < R(s)$ whenever the property does *not* hold (the ranking strictly decreases)

If such an $R$ exists, the property must hold infinitely often — otherwise $R$ would decrease forever below zero, contradicting condition 1.

We train the `RankingNN` to approximate these conditions by:
1. Simulating the counter for $N$ steps (it's a fully functional `gym.Env`)
2. At each step, computing $R(x, y, z)$ and $R(x', y', z')$
3. Penalizing violations: $R$ should decrease at non-target states and be near zero at target states

In [11]:
optimizer = torch.optim.Adam(wrapped_ranking.parameters(), lr=0.01)
margin = 0.1
n_epochs = 300
n_steps = 20

for epoch in range(n_epochs):
    # Simulate the counter to collect a trajectory of (x, y, z) states.
    # CounterEnv is a full gym.Env — reset() and step() work directly.
    wrapped_counter.reset()

    states = []
    for _ in range(n_steps):
        states.append((wrapped_counter.x, wrapped_counter.y, wrapped_counter.z))
        wrapped_counter.step(0)  # Discrete(1) action space — always 0

    # Compute ranking loss over consecutive state pairs
    loss = torch.tensor(0.0)
    for i in range(len(states) - 1):
        s = torch.tensor(states[i], dtype=torch.float32)
        s_next = torch.tensor(states[i + 1], dtype=torch.float32)
        r = wrapped_ranking(s.unsqueeze(0)).squeeze()
        r_next = wrapped_ranking(s_next.unsqueeze(0)).squeeze()

        x, y, z = states[i]
        at_target = (x == y) or (x == z)

        if not at_target:
            # R must strictly decrease by at least `margin`
            loss = loss + torch.relu(r_next - r + margin)
            # R must be non-negative
            loss = loss + torch.relu(-r)
        else:
            # R should be ~0 at target states
            loss = loss + r ** 2

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if epoch % 50 == 0:
        print(f"epoch {epoch:3d}  loss = {loss.item():.4f}")

epoch   0  loss = 5.3115
epoch  50  loss = 1.2980
epoch 100  loss = 0.3205
epoch 150  loss = 0.0823
epoch 200  loss = 0.0604
epoch 250  loss = 0.0610


## Evaluating the trained ranking function

After training, we evaluate $R$ along a trajectory. At non-target states, $R$ should decrease on each step. At target states ($x = y$ or $x = z$), $R$ should be near zero.

In [12]:
wrapped_counter.reset()

print(f"{'step':>4}  {'x':>5} {'y':>5} {'z':>5}  {'R(s)':>8}  {'target':>6}")
print("-" * 45)

with torch.no_grad():
    for step in range(20):
        x, y, z = wrapped_counter.x, wrapped_counter.y, wrapped_counter.z
        s = torch.tensor([x, y, z], dtype=torch.float32)
        r = wrapped_ranking(s.unsqueeze(0)).squeeze().item()
        at_target = (x == y) or (x == z)
        print(f"{step:4d}  {x:5.1f} {y:5.1f} {z:5.1f}  {r:8.4f}  {'  *' if at_target else ''}")
        wrapped_counter.step(0)

step      x     y     z      R(s)  target
---------------------------------------------
   0    0.0   5.0   3.0    0.4156  
   1    1.0   5.0   3.0    0.3146  
   2    2.0   5.0   3.0    0.2135  
   3    3.0   5.0   3.0    0.1125    *
   4    4.0   5.0   3.0    0.0114  
   5    5.0   5.0   3.0   -0.0897    *
   6    0.0   5.0   3.0    0.4156  
   7    1.0   5.0   3.0    0.3146  
   8    2.0   5.0   3.0    0.2135  
   9    3.0   5.0   3.0    0.1125    *
  10    4.0   5.0   3.0    0.0114  
  11    5.0   5.0   3.0   -0.0897    *
  12    0.0   5.0   3.0    0.4156  
  13    1.0   5.0   3.0    0.3146  
  14    2.0   5.0   3.0    0.2135  
  15    3.0   5.0   3.0    0.1125    *
  16    4.0   5.0   3.0    0.0114  
  17    5.0   5.0   3.0   -0.0897    *
  18    0.0   5.0   3.0    0.4156  
  19    1.0   5.0   3.0    0.3146  
